# 06 - Constrained Optimization of Hydrogen-Defect Configurations

## Aim of this notebook

In this notebook, we use optimization to search for hydrogen-defect configurations that maximize the transport-related descriptor.

A direct maximization of the descriptor may lead to a trivial solution with very few weak defects. Therefore, we perform constrained optimization.

The goal is not simply to remove defects, but to identify defect configurations that maintain a meaningful defect density while still producing a high transport-related score.

The optimization problem is:

```text
maximize transport_descriptor
```

subject to constraints on defect number, defect density, defect strength, defect arrangement, and localization behavior.

This makes the optimization more scientifically realistic.


## Mathematical Formulas Used in This Notebook

The optimization problem is to find a defect configuration that maximizes the transport descriptor.

A candidate configuration is

$$
\theta = (m, V, A, D),
$$

where $m$ is the number of defects, $V$ is the defect strength, $A$ is the arrangement mode, and $D$ is the set of defect positions.

The optimization objective is

$$
\theta^* = \arg\max_{\theta \in \Omega} T(\theta),
$$

where $\Omega$ is the allowed search space.

The constraints are

$$
m_{\min} \le m \le m_{\max},
$$

$$
V_{\min} \le V \le V_{\max},
$$

and

$$
A \in \{\mathrm{random},\mathrm{clustered},\mathrm{spread}\}.
$$

The transport descriptor used in the search is

$$
T(\theta)=\frac{1}{1+\alpha\overline{\mathrm{IPR}}(\theta)+\beta\rho(\theta)}.
$$

The best candidate is the one with the largest $T$ value among all sampled configurations:

$$
\theta_{\mathrm{best}} = \arg\max_{k=1,\ldots,K} T(\theta_k),
$$

where $K$ is the number of random-search trials.


## How to read this notebook

This notebook uses the physics model as a search engine.

Instead of only analyzing random configurations, it searches for defect arrangements that maximize the transport descriptor under simple constraints.

The workflow is:

1. Generate many candidate defect configurations.
2. Evaluate each candidate using the same physics functions.
3. Rank the configurations by transport descriptor.
4. Save and interpret the best configuration.

This notebook is a first step toward inverse design: choosing material/defect settings that produce desired transport behavior.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.linalg import eigh

ROOT = Path('..').resolve()
RESULTS_DIR = ROOT / 'results'
FIGURE_DIR = ROOT / 'figures'

RESULTS_DIR.mkdir(exist_ok=True)
FIGURE_DIR.mkdir(exist_ok=True)

### Explanation of the code cell above

This cell imports the libraries and creates output folders.

The optimization notebook needs:

- `numpy` for random search,
- `pandas` for storing candidate configurations,
- `matplotlib` for visualization,
- `eigh` for solving the Hamiltonian eigenvalue problem.

The `results` folder stores optimization tables, and the `figures` folder stores plots.


## Reuse Physics Functions

These functions continue the same one-dimensional tight-binding model used in the previous notebooks.

In [ ]:
def build_clean_hamiltonian(n_sites=40, hopping=1.0, epsilon_0=0.0):
    """Build a clean 1D nearest-neighbor tight-binding Hamiltonian."""
    hamiltonian = np.zeros((n_sites, n_sites), dtype=float)
    np.fill_diagonal(hamiltonian, epsilon_0)

    for i in range(n_sites - 1):
        hamiltonian[i, i + 1] = -hopping
        hamiltonian[i + 1, i] = -hopping

    return hamiltonian


def add_defects_at_sites(hamiltonian, defect_sites, defect_strength=2.0):
    """Add hydrogen-like defects as local diagonal perturbations."""
    defected = hamiltonian.copy()

    for site in defect_sites:
        defected[int(site), int(site)] += defect_strength

    return defected


def compute_ipr(eigenvectors):
    """Compute inverse participation ratio for each eigenstate."""
    return np.sum(np.abs(eigenvectors) ** 4, axis=0)


def defect_density(n_defects, n_sites):
    """Calculate defect density."""
    return n_defects / n_sites


def mean_defect_distance(defect_sites):
    """Calculate the mean distance between neighboring defect sites."""
    defect_sites = sorted(defect_sites)

    if len(defect_sites) < 2:
        return np.nan

    distances = np.diff(defect_sites)
    return float(np.mean(distances))


def clustering_index(defect_sites):
    """Calculate clustering index. Undefined for fewer than two defects."""
    mean_distance = mean_defect_distance(defect_sites)

    if np.isnan(mean_distance):
        return np.nan

    return 1.0 / (1.0 + mean_distance)


def energy_gap(eigenvalues):
    """Calculate central energy gap."""
    n = len(eigenvalues)
    return float(eigenvalues[n // 2] - eigenvalues[n // 2 - 1])


def transport_descriptor(mean_ipr, density, alpha=1.0, beta=1.0):
    """Calculate transport-related descriptor."""
    return 1.0 / (1.0 + alpha * mean_ipr + beta * density)

### Explanation of the code cell above

This cell redefines the physics functions used throughout the project.

The functions build a clean lattice, add defects, compute IPR, and calculate transport descriptors.

They are repeated here so that Notebook 6 can run independently. This makes the notebook easier to share because it does not depend on hidden functions from earlier notebooks.


## Generate Defect Arrangements

We compare three spatial patterns: random, clustered, and spread defects.

In [ ]:
def generate_defect_sites(n_sites, n_defects, mode, rng):
    """Generate defect sites using random, clustered, or spread arrangements."""
    if mode == 'random':
        return sorted(
            rng.choice(
                n_sites,
                size=n_defects,
                replace=False,
            ).tolist()
        )

    if mode == 'clustered':
        start = int(rng.integers(0, n_sites - n_defects + 1))
        return list(range(start, start + n_defects))

    if mode == 'spread':
        return sorted(
            np.linspace(
                0,
                n_sites - 1,
                n_defects,
                dtype=int,
            ).tolist()
        )

    raise ValueError("mode must be 'random', 'clustered', or 'spread'")

### Explanation of the code cell above

This cell defines how candidate defect arrangements are generated.

The optimization search can test three arrangement modes:

- `random`: defect sites are chosen randomly,
- `clustered`: defects are placed close together,
- `spread`: defects are distributed across the lattice.

These modes allow the search to compare different physical design strategies.


## Evaluate One Configuration

Each candidate configuration becomes one row in the optimization table.

In [ ]:
def evaluate_configuration(
    n_sites,
    hopping,
    epsilon_0,
    n_defects,
    defect_strength,
    arrangement_mode,
    rng,
):
    """Build and evaluate one defect configuration."""
    H_clean = build_clean_hamiltonian(
        n_sites=n_sites,
        hopping=hopping,
        epsilon_0=epsilon_0,
    )

    defect_sites = generate_defect_sites(
        n_sites=n_sites,
        n_defects=n_defects,
        mode=arrangement_mode,
        rng=rng,
    )

    H_defected = add_defects_at_sites(
        H_clean,
        defect_sites=defect_sites,
        defect_strength=defect_strength,
    )

    eigenvalues, eigenvectors = eigh(H_defected)
    ipr_values = compute_ipr(eigenvectors)

    mean_ipr = float(np.mean(ipr_values))
    max_ipr = float(np.max(ipr_values))
    density = defect_density(n_defects, n_sites)
    gap = energy_gap(eigenvalues)
    bandwidth = float(np.max(eigenvalues) - np.min(eigenvalues))
    mean_distance = mean_defect_distance(defect_sites)
    cluster_score = clustering_index(defect_sites)

    t_score = transport_descriptor(
        mean_ipr=mean_ipr,
        density=density,
        alpha=1.0,
        beta=1.0,
    )

    return {
        'n_sites': n_sites,
        'n_defects': n_defects,
        'defect_density': density,
        'defect_strength': defect_strength,
        'arrangement_mode': arrangement_mode,
        'defect_sites': ';'.join(map(str, defect_sites)),
        'mean_defect_distance': mean_distance,
        'clustering_index': cluster_score,
        'energy_gap': gap,
        'energy_bandwidth': bandwidth,
        'mean_ipr': mean_ipr,
        'max_ipr': max_ipr,
        'transport_descriptor': t_score,
    }

### Mathematical meaning of one configuration evaluation

For each candidate configuration $\theta$, the notebook computes

$$
T(\theta)=\frac{1}{1+\alpha\overline{\mathrm{IPR}}(\theta)+\beta\rho(\theta)}.
$$

This single value summarizes whether the candidate configuration is expected to support better or worse transport.


### Explanation of the code cell above

This cell defines how one candidate configuration is evaluated.

For a proposed defect configuration, the function:

1. builds the clean Hamiltonian,
2. generates defect sites,
3. adds the defects,
4. solves the eigenvalue problem,
5. computes localization and transport descriptors,
6. returns all relevant information as a dictionary.

This function is the core evaluator used by the random search.


## Constrained Random Search

We generate candidate configurations subject to the constraints:

```text
3 <= n_defects <= 10
0.5 <= defect_strength <= 4.0
arrangement_mode in {random, clustered, spread}
```

In [ ]:
rng = np.random.default_rng(42)

n_trials = 2000
n_sites = 40
hopping = 1.0
epsilon_0 = 0.0

min_defects = 3
max_defects = 10

min_strength = 0.5
max_strength = 4.0

arrangement_modes = ['random', 'clustered', 'spread']
records = []

for trial in range(n_trials):
    n_defects = int(rng.integers(min_defects, max_defects + 1))
    defect_strength = float(rng.uniform(min_strength, max_strength))
    arrangement_mode = str(rng.choice(arrangement_modes))

    result = evaluate_configuration(
        n_sites=n_sites,
        hopping=hopping,
        epsilon_0=epsilon_0,
        n_defects=n_defects,
        defect_strength=defect_strength,
        arrangement_mode=arrangement_mode,
        rng=rng,
    )

    result['trial'] = trial
    records.append(result)

optimization_results = pd.DataFrame(records)
optimization_results.head()

### Mathematical meaning of random search

Random search samples $K$ candidate configurations

$$
\theta_1,\theta_2,\ldots,\theta_K \sim p(\theta),
$$

then chooses

$$
\theta_{\mathrm{best}} = \arg\max_{k=1,\ldots,K} T(\theta_k).
$$


### Explanation of the code cell above

This cell performs constrained random search.

For each trial, the code randomly chooses:

- number of defects,
- defect strength,
- arrangement mode.

Then it evaluates the candidate and stores the result.

The constraints are the allowed ranges for defect count and defect strength. This is called constrained search because the algorithm does not test impossible or unwanted configurations.


## Find and Save the Best Configuration

In [ ]:
best_result = optimization_results.sort_values(
    'transport_descriptor',
    ascending=False,
).iloc[0]

optimization_results.to_csv(
    RESULTS_DIR / 'notebook6_constrained_random_search_results.csv',
    index=False,
)

pd.DataFrame([best_result]).to_csv(
    RESULTS_DIR / 'notebook6_best_configuration.csv',
    index=False,
)

best_result

### Explanation of the code cell above

This cell finds the best configuration from the random search.

The best result is the row with the highest `transport_descriptor`.

The full optimization table and the best configuration are saved as CSV files. Saving both is important: the best row gives the final answer, while the full table allows later analysis of all candidates.


## Interpretation of the Best Configuration

The best configuration is the one that maximizes the transport-related descriptor under the imposed constraints.

Because the descriptor favors low localization and low defect density, the optimization was constrained to avoid the trivial solution of selecting nearly zero defects or extremely weak perturbations.

The result should be interpreted as:

> Within the constrained design space, this configuration gives the highest transport-related score.

It should not be interpreted as:

> This is the globally optimal real material configuration.

The current optimization is a proof-of-concept search within a simplified one-dimensional tight-binding model.

## Plot Optimization Results

In [ ]:
plt.figure(figsize=(7, 5))
plt.hist(
    optimization_results['transport_descriptor'],
    bins=40,
    alpha=0.8,
)
plt.xlabel('Transport descriptor')
plt.ylabel('Count')
plt.title('Distribution of transport descriptor values')
plt.grid(True)
plt.tight_layout()
plt.savefig(FIGURE_DIR / 'notebook6_transport_descriptor_distribution.png', dpi=200)
plt.show()

plt.figure(figsize=(7, 5))
optimization_results.boxplot(
    column='transport_descriptor',
    by='arrangement_mode',
)
plt.xlabel('Arrangement mode')
plt.ylabel('Transport descriptor')
plt.title('Transport descriptor by defect arrangement')
plt.suptitle('')
plt.grid(True)
plt.tight_layout()
plt.savefig(FIGURE_DIR / 'notebook6_transport_by_arrangement.png', dpi=200)
plt.show()

plt.figure(figsize=(7, 5))
scatter = plt.scatter(
    optimization_results['defect_strength'],
    optimization_results['transport_descriptor'],
    c=optimization_results['defect_density'],
    alpha=0.7,
)
plt.colorbar(scatter, label='Defect density')
plt.xlabel('Defect strength')
plt.ylabel('Transport descriptor')
plt.title('Optimization search: strength vs transport')
plt.grid(True)
plt.tight_layout()
plt.savefig(FIGURE_DIR / 'notebook6_strength_vs_transport.png', dpi=200)
plt.show()

### Explanation of the code cell above

This cell visualizes the optimization results.

The histogram shows the distribution of transport-descriptor values across all tested configurations. The boxplot compares transport behavior across arrangement modes.

These plots help you understand whether the best result is exceptional or whether many configurations perform similarly.


## Conclusion

In this notebook, we performed constrained optimization of hydrogen-defect configurations.

Because the transport descriptor favors lower defect density and lower localization, direct unconstrained maximization may lead to trivial solutions. Therefore, the search was constrained to meaningful ranges of defect number and defect strength.

The constrained random search identified the best configuration within the chosen design space. The result should be interpreted as a proof-of-concept optimization in a simplified one-dimensional tight-binding model, not as a real material discovery.

The next step is to extend this optimization using Optuna and then compare selected representative cases, such as clean, weak random, strong random, and clustered defect configurations.

## Key learning takeaway

After running this notebook, focus on writing one or two sentences in your own words:

- What physical quantity was computed?
- Which feature or parameter changed?
- How did the result affect localization or transport?

This habit will help you turn code execution into scientific understanding.


## Final Formula Reminder

The core logic of this notebook can be summarized as

$$
\text{defect configuration} \rightarrow H \rightarrow \{E_n,\mathbf{c}^{(n)}\} \rightarrow \text{features} \rightarrow \text{prediction or optimization}.
$$

This is the main physics-informed workflow: we start from a material-inspired defect model, convert it into spectral and localization descriptors, and then use those descriptors for machine learning, interpretation, or optimization.
